In [ ]:
ROLE  = "relay"
import os,time, pymongo ,copy
import numpy as np
import matplotlib.pyplot as plt
import hmac


import sys
sys.path.append('../../')

import src
conf = src.CONFIG()
rx = src.RX(role=ROLE,conf=conf)

In [ ]:
def run():
    global cnt_good, cnt_bad, collection
    src.MQTT_RX(conf=conf, role=ROLE).send_ready_and_wait_for_begin()
    file = rx.record()

    demod = src.rx.Demodulation(conf=conf)
    pp = src.PostProcessing(file=file, conf=conf, demod=demod, role=ROLE, plot=False)

    if pp.check():
        print("Recording is correct")
        for i in range(len(pp.Frames)):
            frame = pp.frameByNumber(i)
            hard_decision,rs, SNR = demod.decode(frame)
            index = demod.detect_message_indices(received=list(hard_decision), preamble=conf.PREAMBLE, repeat=conf.PREAMBLE_REPEAT)
            if index[0] is None or index[1] is None:
                print("No preamble detected!")
                continue

            msg_hard_decision = hard_decision[index[0]:index[1]]
            print("Message: ", pp.bits_to_string(msg_hard_decision[0:-256]))
            # print("recieved MAC: ", pp.binary_list_to_hex(msg_hard_decision[-256:]))
            expected_MAC = hmac.new(conf.MAC_KEY.encode('utf-8'), msg=pp.bits_to_string(msg_hard_decision[0:-256]).encode('utf-8'), digestmod='sha256').hexdigest()
            if pp.binary_list_to_hex(msg_hard_decision[-256:]) == expected_MAC:
                print("Good MAC")
                mac = True
                cnt_good += 1
            else:
                print("MAC is not correct")
                mac = False
                cnt_bad += 1

            SNR = SNR[index[0]+10:index[1]-10]
            print("SNR: ", np.nanmean(SNR))

            insert={
                'SNR': float(np.nanmean(SNR)),
                'MAC': pp.binary_list_to_hex(msg_hard_decision[-256:]),
                'message': pp.bits_to_string(msg_hard_decision[0:-256]),
                'integrity': mac,
                'time': time.time(),
                'config': copy.deepcopy(conf.config)
            }
            collection.insert_one(insert)

cnt_good = 0
cnt_bad = 0

myclient = pymongo.MongoClient("mongodb://localhost:27017/")
mydb = myclient["MAC_SUPERPOSITION"]
collection = mydb[f'1D_{ROLE}_phase_1']
while True:
    print(fr"{cnt_good=}, {cnt_bad=}")
    run()